# LightGBM Training on Dataset A

In this notebook, we train a LightGBM model using Dataset A features.

The goal is to:
- train a baseline model
- evaluate performance using AUROC and AUPRC
- establish a reference for future feature comparisons (to also see which set of features wors best for this problem)

In [16]:
import pandas as pd
import numpy as np

import lightgbm as lgb

from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score

In [17]:
train_df = pd.read_csv("../data/dataset_A_train.csv")
val_df = pd.read_csv("../data/dataset_A_val.csv")
test_df = pd.read_csv("../data/dataset_A_test.csv")

print(train_df.shape, val_df.shape, test_df.shape)

(5718, 225) (1439, 225) (1812, 225)


## Prep Features

We separate features and labels and remove non-feature columns such as peptide and HLA identifiers.

In [19]:
drop_cols = ["peptide", "HLA", "hla_sequence", "index"]

# split features and labels
X_train = train_df.drop(columns=drop_cols + ["Label"], errors="ignore")
y_train = train_df["Label"]

X_val = val_df.drop(columns=drop_cols + ["Label"], errors="ignore")
y_val = val_df["Label"]

X_test = test_df.drop(columns=drop_cols + ["Label"], errors="ignore")
y_test = test_df["Label"]

# 🔹 combine to ensure consistent encoding
X_all = pd.concat([X_train, X_val, X_test])

# find categorical (object) columns
cat_cols = X_all.select_dtypes(include=["object"]).columns

# convert to category + encode
for col in cat_cols:
    X_all[col] = X_all[col].astype("category")
    X_all[col] = X_all[col].cat.codes

# split back
X_train = X_all.iloc[:len(X_train)]
X_val   = X_all.iloc[len(X_train):len(X_train)+len(X_val)]
X_test  = X_all.iloc[len(X_train)+len(X_val):]

print(X_train.shape)

(5718, 220)


## Model Training

We train a LightGBM classifier using the training set and monitor performance on the validation set.

In [20]:
model = lgb.LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    callbacks=[lgb.log_evaluation(10)]
)

[LightGBM] [Info] Number of positive: 2591, number of negative: 3127
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005331 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 35424
[LightGBM] [Info] Number of data points in the train set: 5718, number of used features: 220
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.453130 -> initscore=-0.188030
[LightGBM] [Info] Start training from score -0.188030
[10]	valid_0's auc: 0.844118	valid_0's binary_logloss: 0.566466
[20]	valid_0's auc: 0.850876	valid_0's binary_logloss: 0.51275
[30]	valid_0's auc: 0.854737	valid_0's binary_logloss: 0.485345
[40]	valid_0's auc: 0.855769	valid_0's binary_logloss: 0.472119
[50]	valid_0's auc: 0.858058	valid_0's binary_logloss: 0.464284
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[60]	valid_0's auc: 0.860627	valid_0's

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,6
,learning_rate,0.05
,n_estimators,300
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [21]:
y_pred_proba = model.predict_proba(X_test)[:, 1]

from sklearn.metrics import roc_auc_score
print("Test AUC:", roc_auc_score(y_test, y_pred_proba))

Test AUC: 0.8627298275668425


In [22]:
X_train.tail(20)


,Peptide_AF1,Peptide_AF2,Peptide_AF3,Peptide_AF4,Peptide_AF5,Peptide_BLOSUM1,Peptide_BLOSUM2,Peptide_BLOSUM3,Peptide_BLOSUM4,Peptide_BLOSUM5,...,HLA_Z4,HLA_Z5,HLA_boman,HLA_hydrophobicity,HLA_charge,HLA_molecular_weight,HLA_aliphatic_index,HLA_instability_index,HLA_isoelectric_point,HLA_mz
5698,-0.920741,-0.334062,0.515995,0.866040,-0.164885,-0.972000,-0.795000,-0.479000,0.157000,-0.343000,...,0.214706,-0.317941,0.506176,-0.120588,0.265832,4282.84454,68.823529,-4.388235,7.749856,2141.021991
5699,-0.704815,-0.421679,-0.966078,0.283048,-0.698477,-0.632222,-0.407778,-0.198889,-0.256667,0.077778,...,-0.321176,-0.158235,1.232941,-0.341176,-0.007454,4361.83554,74.705882,34.447059,6.440692,2180.538736
5700,-0.819970,0.162623,-0.570303,0.794227,-0.566142,-0.621111,-0.584444,-0.173333,0.035556,-0.431111,...,0.214706,-0.317941,0.506176,-0.120588,0.265832,4282.84454,68.823529,-4.388235,7.749856,2141.021991
5701,-0.704647,-0.234281,0.851792,0.647056,0.370558,-0.678889,-0.531111,-0.223333,0.070000,-0.215556,...,-0.321176,-0.158235,1.232941,-0.341176,-0.007454,4361.83554,74.705882,34.447059,6.440692,2180.538736
5702,0.466389,0.685202,0.344713,0.238035,1.027813,0.635000,0.211000,0.054000,-0.015000,0.149000,...,-0.321176,-0.158235,1.232941,-0.341176,-0.007454,4361.83554,74.705882,34.447059,6.440692,2180.538736
5703,-0.078525,-0.117789,-0.452439,0.557689,-0.437548,0.048000,-0.275000,-0.152000,0.002000,-0.187000,...,0.214706,-0.317941,0.506176,-0.120588,0.265832,4282.84454,68.823529,-4.388235,7.749856,2141.021991
5704,-0.207290,0.493194,-0.888685,0.536794,-0.659111,0.315556,-0.461111,-0.024444,0.138889,-0.305556,...,0.214706,-0.317941,0.506176,-0.120588,0.265832,4282.84454,68.823529,-4.388235,7.749856,2141.021991
5705,-0.575756,-0.242406,-0.582472,0.909913,-0.497233,-0.118889,-0.852222,0.037778,-0.107778,-0.140000,...,-0.321176,-0.158235,1.232941,-0.341176,-0.007454,4361.83554,74.705882,34.447059,6.440692,2180.538736
5706,-0.524586,-0.289469,-1.027921,0.594400,-0.589687,-0.247000,-0.434000,-0.050000,0.196000,0.128000,...,0.214706,-0.317941,0.506176,-0.120588,0.265832,4282.84454,68.823529,-4.388235,7.749856,2141.021991
5707,-0.325619,-0.356342,-0.482359,0.644054,-0.063059,-0.137778,-0.262222,-0.202222,0.195556,0.134444,...,0.214706,-0.317941,0.506176,-0.120588,0.265832,4282.84454,68.823529,-4.388235,7.749856,2141.021991


In [23]:
# basuc feature importance

importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values(by="importance", ascending=False)

print(importance.head(20))

                       feature  importance
14            Peptide_BLOSUM10         151
18                  Peptide_F1         133
33                Peptide_KF10         128
53             Peptide_ProtFP7         127
54             Peptide_ProtFP8         106
107  Peptide_instability_index         106
13             Peptide_BLOSUM9         105
32                 Peptide_KF9         104
8              Peptide_BLOSUM4         102
26                 Peptide_KF3          99
105   Peptide_molecular_weight          95
51             Peptide_ProtFP5          95
2                  Peptide_AF3          92
100                 Peptide_Z4          89
55                 Peptide_SV1          88
30                 Peptide_KF7          84
99                  Peptide_Z3          83
72              Peptide_SVGER6          82
16                 Peptide_PP2          82
77             Peptide_SVGER11          82
